# Bates model calibration to SPX options

We calibrate the Bates (1996) stochastic volatility model to live S&P 500 option prices. Under the risk-neutral measure $\mathbb{Q}$, the dynamics are given by:

$\large \mathrm{d}S_t = (r - q - \lambda\bar{k})\, S_t\, \mathrm{d}t + \sqrt{v_t}\, S_t\, \mathrm{d}W_t^S + (e^J - 1)\, S_t\, \mathrm{d}N_t$

$\large \mathrm{d}v_t = \kappa(\theta - v_t)\, \mathrm{d}t + \xi\sqrt{v_t}\, \mathrm{d}W_t^v$

$\large \text{Cov}(\mathrm{d}W_t^S, \mathrm{d}W_t^v) = \rho\, \mathrm{d}t, \quad J \sim \mathcal{N}(\mu_j, \sigma_j^2), \quad N_t \sim \text{Poisson}(\lambda)$

where $\bar{k} = e^{\mu_j + \frac{1}{2}\sigma_j^2} - 1$ is the compensator term. Four calibration approaches are compared: 
1. Semi-analytical naive midpoint Albrecher (2007) integration
2. Semi-analytical cached Gauss-Legendre quadrature
3. Full truncation Monte Carlo (Lord et al. 2010), with CRN
4. Quadratic exponential Monte Carlo (Andersen 2008), with slice pricing

In [1]:
import os, json, time, logging
import numpy as np
import pandas as pd
from datetime import datetime
from scipy.optimize import minimize

from batespricer.calibration import (
    BatesCalibrator, BatesCalibratorFast,
    BatesCalibratorMC, BatesCalibratorMCFast
)
from batespricer.analytics import BatesAnalyticalPricer, implied_volatility
from batespricer.data import (
    fetch_treasury_rates_fred, fetch_raw_data, fetch_options,
    get_market_implied_spot, ImpliedDividendCurve,
    save_options_to_cache
)

logging.basicConfig(level=logging.INFO, format='%(message)s')

PARAM_KEYS = ['kappa', 'theta', 'xi', 'rho', 'v0', 'lamb', 'mu_j', 'sigma_j']

In [2]:
def print_curves(r_curve, q_curve):
    print('\n' + '='*60)
    print(f"{'Tenor':<10} | {'Rate (r)':<15} | {'Yield (q)':<15}")
    print('-'*60)
    for t, label in [(0.02,'1w'),(0.04,'2w'),(0.0833,'1M'),(0.25,'3M'),(0.5,'6M'),(1.0,'1Y')]:
        print(f"{label:<10} | {r_curve.get_rate(t)*100:>13.4f}% | {q_curve.get_rate(t)*100:>13.4f}%")
    print('='*60 + '\n')

def validate(ticker, S0, r_curve, q_curve, params, options, tag=''):
    p = [params[k] for k in PARAM_KEYS] if isinstance(params, dict) else list(params)
    kappa, theta, xi, rho, v0, lamb, mu_j, sigma_j = p

    strikes  = np.array([o.strike for o in options])
    mats     = np.array([o.maturity for o in options])
    types    = np.array([o.option_type for o in options])
    r_T      = np.array([r_curve.get_rate(o.maturity) for o in options])
    q_T      = np.array([q_curve.get_rate(o.maturity) for o in options])

    model_prices = BatesAnalyticalPricer.price_vectorized(
        S0, strikes, mats, r_T, q_T, types,
        kappa, theta, xi, rho, v0, lamb, mu_j, sigma_j
    )

    rows = []
    for i, opt in enumerate(options):
        mp = model_prices[i]
        bid = getattr(opt, 'bid', 0.0)
        ask = getattr(opt, 'ask', 0.0)
        spread = ask - bid if ask > 0 and bid > 0 else 0.0
        try:
            rv, qv = r_curve.get_rate(opt.maturity), q_curve.get_rate(opt.maturity)
            iv_mkt   = implied_volatility(opt.market_price, S0, opt.strike, opt.maturity, rv, qv, opt.option_type)
            iv_model = implied_volatility(mp, S0, opt.strike, opt.maturity, rv, qv, opt.option_type)
        except Exception:
            iv_mkt, iv_model = 0.0, 0.0

        rows.append({
            'T': round(opt.maturity, 4), 'K': opt.strike, 'Type': opt.option_type,
            'Spread': round(spread, 5), 'Market': round(opt.market_price, 4),
            'Model': round(mp, 4), 'Err': round(mp - opt.market_price, 4),
            'IV_Mkt': round(iv_mkt, 4), 'IV_Model': round(iv_model, 4)
        })

    df = pd.DataFrame(rows)
    rmse = float(np.sqrt((df['Err']**2).mean()))

    print(f"\n{'='*80}")
    print(f"{tag} | S0: {S0:.2f} | RMSE: {rmse:.4f}")
    print('-'*80)
    print(df[['T','K','Type','Market','Model','Spread','IV_Mkt']].head(5).to_string(index=False))
    print('='*80)

    os.makedirs('results', exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    base = f'results/calibration_{tag}_{ticker}_{ts}'
    df.to_csv(f'{base}_prices.csv', index=False)

    tenors = [(0.02,'1w'),(0.04,'2w'),(0.0833,'1M'),(0.25,'3M'),(0.5,'6M'),(1.0,'1Y')]
    meta = {
        'model': f'Bates_{tag}',
        'market': {
            'S0': S0,
            'r_sample': {f'{t:.4f}Y': float(r_curve.get_rate(t)) for t,_ in tenors},
            'q_sample': {f'{t:.4f}Y': float(q_curve.get_rate(t)) for t,_ in tenors},
        },
        'analytical': {**dict(zip(PARAM_KEYS, p)), 'rmse': rmse}
    }
    with open(f'{base}_meta.json', 'w') as f:
        json.dump(meta, f, indent=4)

    return rmse

## Market data

We use the FRED UST yield curve (NSS-OLS fit) for the risk-free rate, fetch  $ \ \approx 300$ OTM options from yfinance, and extract the implied dividend curve from put-call parity.

In [3]:
ticker = "^SPX"
target_date = datetime.now().strftime("%Y-%m-%d")
print(f"date: ")
print(target_date)

r_curve = fetch_treasury_rates_fred(target_date, os.getenv("FRED_API_KEY"))
raw_df  = fetch_raw_data(ticker)
S0      = get_market_implied_spot(ticker, raw_df, r_curve)
options = fetch_options(ticker, S0, target_size=300)
save_options_to_cache(options, ticker)

q_curve = ImpliedDividendCurve(raw_df, S0, r_curve, ticker)
print_curves(r_curve, q_curve)
print(f"{len(options)} options loaded (S0 = {S0:.2f})")

date: 
2026-03-01

Tenor      | Rate (r)        | Yield (q)      
------------------------------------------------------------
1w         |        4.2160% |        1.7208%
2w         |        4.2076% |        1.5943%
1M         |        4.1900% |        1.3016%
3M         |        4.1301% |        1.0518%
6M         |        4.0600% |        1.0500%
1Y         |        3.9700% |        1.0753%

300 options loaded (S0 = 6864.99)


## Part 1 - Semi-analytical calibration (naive)

We use the Albrecher (2007) single-integral formulation with midpoint quadrature on a quadratically transformed $n = 296$ node grid. The calibration is formulated as spread-weighted nonlinear least squares:

$\large \min_{\mathbf{p} \in \Theta} \sum_{i=1}^{N} w_i \left(c_i^{\text{model}}(\mathbf{p}) - c_i^{\text{mid}}\right)^2, \quad w_i = \frac{1}{\left(c_i^{\text{ask}} - c_i^{\text{bid}}\right)^2}$

Solved with L-BFGS-B.

In [4]:
t0 = time.time()
res_naive = BatesCalibrator(S0=S0, r_curve=r_curve, q_curve=q_curve).calibrate(options)
print(f"\nDone in {time.time()-t0:.2f}s | RMSE: {res_naive['rmse']:.4f}")
for k in PARAM_KEYS:
    print(f"  {k:<8}: {res_naive[k]:.4f}")

validate(ticker, S0, r_curve, q_curve, res_naive, options, tag='Ana_naive')

[Step] W-Obj: 80.2986 | k:0.9 th:0.143 xi:2.50 rho:-0.54 v0:0.12 | L:0.28 muJ:0.19 sJ:0.12
[Step] W-Obj: 41.2967 | k:0.6 th:0.091 xi:3.42 rho:-0.71 v0:0.08 | L:0.18 muJ:0.30 sJ:0.08
[Step] W-Obj: 34.5817 | k:0.5 th:0.087 xi:3.66 rho:-0.75 v0:0.04 | L:0.12 muJ:0.30 sJ:0.07
[Step] W-Obj: 29.5959 | k:0.6 th:0.101 xi:3.41 rho:-0.71 v0:0.05 | L:0.14 muJ:0.26 sJ:0.08
[Step] W-Obj: 25.8918 | k:0.7 th:0.123 xi:3.11 rho:-0.66 v0:0.06 | L:0.16 muJ:0.21 sJ:0.09
[Step] W-Obj: 23.4341 | k:0.9 th:0.166 xi:2.55 rho:-0.56 v0:0.05 | L:0.19 muJ:0.10 sJ:0.10
[Step] W-Obj: 17.8500 | k:0.8 th:0.160 xi:2.68 rho:-0.59 v0:0.04 | L:0.17 muJ:0.11 sJ:0.10
[Step] W-Obj: 14.9543 | k:0.9 th:0.186 xi:2.44 rho:-0.55 v0:0.02 | L:0.16 muJ:0.04 sJ:0.10
[Step] W-Obj: 14.3341 | k:0.9 th:0.191 xi:2.37 rho:-0.54 v0:0.03 | L:0.16 muJ:0.03 sJ:0.10
[Step] W-Obj: 14.1815 | k:0.9 th:0.186 xi:2.42 rho:-0.55 v0:0.03 | L:0.16 muJ:0.04 sJ:0.10
[Step] W-Obj: 13.5272 | k:0.9 th:0.184 xi:2.42 rho:-0.56 v0:0.03 | L:0.15 muJ:0.02 sJ:0.09


Done in 35.65s | RMSE: 6.9639
  kappa   : 1.2848
  theta   : 0.0678
  xi      : 1.0694
  rho     : -0.7938
  v0      : 0.0247
  lamb    : 0.5334
  mu_j    : -0.1222
  sigma_j : 0.0489

Ana_naive | S0: 6864.99 | RMSE: 6.9639
--------------------------------------------------------------------------------
     T      K Type  Market   Model  Spread  IV_Mkt
0.0414 5400.0  PUT   1.475  4.5392    0.35  0.4869
0.0414 6175.0  PUT   7.150  6.9051    0.70  0.3067
0.0414 6550.0  PUT  26.600 25.5811    0.80  0.2356
0.0414 6690.0  PUT  45.850 44.1528    0.90  0.2087
0.0441 5800.0  PUT   3.450  3.6155    0.70  0.3875


6.963885902755731

## Part 2 - Semi-analytical calibration (accelerated)

Applies Gauss-Legendre quadrature and maturity-based characteristic function caching ($n=956$). This reduces characteristic evaluations, computing once across strikes sharing the same $T$.

In [5]:
t0 = time.time()

res_fast = BatesCalibratorFast(S0=S0, r_curve=r_curve, q_curve=q_curve).calibrate(options)
print(f"\nDone in {time.time()-t0:.2f}s | RMSE: {res_fast['rmse']:.4f}")
for k in PARAM_KEYS:
    print(f"  {k:<8}: {res_fast[k]:.4f}")

validate(ticker, S0, r_curve, q_curve, res_fast, options, tag='Ana_accel')

[Step] W-Obj: 66.6977 | k:0.1 th:0.001 xi:5.00 rho:-0.99 v0:0.00 | L:0.00 muJ:0.50 sJ:0.01
[Step] W-Obj: 50.7071 | k:0.1 th:0.001 xi:4.94 rho:-0.98 v0:0.07 | L:0.00 muJ:0.50 sJ:0.01
[Step] W-Obj: 42.6731 | k:0.2 th:0.007 xi:4.80 rho:-0.95 v0:0.11 | L:0.03 muJ:0.50 sJ:0.01
[Step] W-Obj: 40.6956 | k:0.2 th:0.013 xi:4.73 rho:-0.93 v0:0.11 | L:0.03 muJ:0.50 sJ:0.01
[Step] W-Obj: 37.6154 | k:0.3 th:0.039 xi:4.49 rho:-0.86 v0:0.07 | L:0.05 muJ:0.50 sJ:0.03
[Step] W-Obj: 31.0668 | k:0.6 th:0.104 xi:3.53 rho:-0.70 v0:0.09 | L:0.06 muJ:0.29 sJ:0.08
[Step] W-Obj: 26.2839 | k:0.8 th:0.149 xi:2.92 rho:-0.58 v0:0.08 | L:0.06 muJ:0.16 sJ:0.12
[Step] W-Obj: 25.2716 | k:0.8 th:0.161 xi:2.75 rho:-0.55 v0:0.07 | L:0.06 muJ:0.12 sJ:0.14
[Step] W-Obj: 15.3711 | k:0.9 th:0.187 xi:2.40 rho:-0.49 v0:0.04 | L:0.05 muJ:0.04 sJ:0.16
[Step] W-Obj: 13.7810 | k:0.9 th:0.183 xi:2.44 rho:-0.50 v0:0.03 | L:0.05 muJ:0.04 sJ:0.16
[Step] W-Obj: 13.3561 | k:0.9 th:0.178 xi:2.48 rho:-0.51 v0:0.03 | L:0.04 muJ:0.03 sJ:0.16


Done in 210.35s | RMSE: 7.2940
  kappa   : 1.4689
  theta   : 0.0490
  xi      : 0.7146
  rho     : -0.7361
  v0      : 0.0267
  lamb    : 0.0525
  mu_j    : -0.4997
  sigma_j : 0.2263

Ana_accel | S0: 6864.99 | RMSE: 8.8487
--------------------------------------------------------------------------------
     T      K Type  Market   Model  Spread  IV_Mkt
0.0414 5400.0  PUT   1.475  7.0302    0.35  0.4869
0.0414 6175.0  PUT   7.150  6.2457    0.70  0.3067
0.0414 6550.0  PUT  26.600 19.3474    0.80  0.2356
0.0414 6690.0  PUT  45.850 38.1211    0.90  0.2087
0.0441 5800.0  PUT   3.450  6.0297    0.70  0.3875


8.848696923820668

## Part 3 - Monte Carlo calibration (naive full truncation)

Full truncation Euler discretization (Lord et al. 2010) with $N = 5000$ paths. We use the SLSQP solver since the MC objective surface is noisier. $v_0$ is locked from the shortest-maturity ATM implied vol to anchor the term structure.

In [ ]:
mc_calib = BatesCalibratorMC(S0=S0, r_curve=r_curve, q_curve=q_curve, n_paths=5000, n_steps=1000)
mc_calib._precompute(options)

def mc_obj(p):
    try:
        model_p, market_p, weights = mc_calib.get_prices(p)
        return np.sqrt(np.mean(((model_p - market_p) * weights)**2))
    except Exception:
        return 1e10

def mc_cb(xk):
    print(
        f"  [MC-Iter] Obj: {mc_obj(xk):10.4f} | "
        f"ka:{xk[0]:.3f} th:{xk[1]:.4f} "
        f"xi:{xk[2]:.3f} rho:{xk[3]:.2f} v0:{xk[4]:.4f} | "
        f"lamb:{xk[5]:.4f} mu_j:{xk[6]:.4f} sigma_j:{xk[7]:.4f}"
    )
t0 = time.time()
x0 = [1.5, 0.25, 0.6, -0.2, 0.21, 0.5, -0.05, 0.2]
bounds = [
    (1.0, 5.0), (0.001, 0.5), (0.01, 0.9), (-0.99, 0.0),
    (0.001, 0.1), (0.0, 0.5), (-0.3, 0.0), (0.05, 0.3)
]

res_mc = minimize(mc_obj, x0, method='SLSQP', bounds=bounds, callback=mc_cb,
                  tol=1e-8, options={'maxiter': 500, 'eps': 3e-3})

final_mod_p, final_mkt_p, _ = mc_calib.get_prices(res_mc.x)
final_rmse = np.sqrt(np.mean((final_mod_p - final_mkt_p)**2))
res_mc_params = dict(zip(PARAM_KEYS, res_mc.x))
res_mc_params['rmse'] = final_rmse
res_mc_params['weighted_obj'] = res_mc.fun

print(f"\nFinal Standard RMSE: {final_rmse:.4f}")
for k in PARAM_KEYS:
    print(f"  {k:<8}: {res_mc_params[k]:.4f}")

validate(ticker, S0, r_curve, q_curve, res_mc_params, options, tag='MC_naive')

LOCK v0: 0.0424 (Vol: 20.6%)


c:\Users\luucv\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\optimize\_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
c:\Users\luucv\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\optimize\_slsqp_py.py:439: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)


  [MC-Iter] Obj:    65.2119 | ka:1.000 th:0.0010 xi:0.900 rho:-0.99 v0:0.0010 | lamb:0.0000 mu_j:0.0000 sigma_j:0.0500
  [MC-Iter] Obj:    59.2872 | ka:1.000 th:0.0471 xi:0.900 rho:-0.99 v0:0.1000 | lamb:0.0000 mu_j:0.0000 sigma_j:0.0500
  [MC-Iter] Obj:   158.6818 | ka:1.000 th:0.5000 xi:0.900 rho:0.00 v0:0.0613 | lamb:0.5000 mu_j:-0.0423 sigma_j:0.1670
  [MC-Iter] Obj:    62.6608 | ka:1.000 th:0.0010 xi:0.900 rho:-0.14 v0:0.0010 | lamb:0.5000 mu_j:-0.0000 sigma_j:0.0500
  [MC-Iter] Obj:    40.8911 | ka:1.757 th:0.0482 xi:0.010 rho:-0.20 v0:0.0010 | lamb:0.0000 mu_j:-0.3000 sigma_j:0.0500
  [MC-Iter] Obj:    71.5696 | ka:1.000 th:0.1471 xi:0.900 rho:-0.62 v0:0.0010 | lamb:0.5000 mu_j:-0.3000 sigma_j:0.3000
  [MC-Iter] Obj:    87.5031 | ka:1.473 th:0.0949 xi:0.900 rho:-0.09 v0:0.1000 | lamb:0.0000 mu_j:-0.2999 sigma_j:0.2999
  [MC-Iter] Obj:    74.6376 | ka:1.000 th:0.1599 xi:0.900 rho:-0.73 v0:0.0010 | lamb:0.5000 mu_j:-0.3000 sigma_j:0.2978
  [MC-Iter] Obj:    24.1818 | ka:1.000 th:0

11.678184994538606

## Part 4 - Monte Carlo calibration (accelerated QE)

Quadratic exponential scheme (Andersen 2008) for the variance process, with the maturity-cached European option evaluation. This reduces the number of evaluations. We use L-BFGS-B since the QE surface is smoother.

In [15]:
t0 = time.time()
res_mc_fast = BatesCalibratorMCFast(S0=S0, r_curve=r_curve, q_curve=q_curve).calibrate(options)
print(f"\nDone in {time.time()-t0:.2f}s | RMSE: {res_mc_fast['rmse']:.4f}")
for k in PARAM_KEYS:
    print(f"  {k:<8}: {res_mc_fast[k]:.4f}")

validate(ticker, S0, r_curve, q_curve, res_mc_fast, options, tag='MC_accel')

[Step] W-Obj: 18.8267 | k:1.1 th:0.049 xi:0.84 rho:-0.29 v0:0.02 | L:0.10 muJ:-0.01 sJ:0.08
[Step] W-Obj: 18.5907 | k:1.1 th:0.057 xi:0.83 rho:-0.28 v0:0.02 | L:0.11 muJ:-0.01 sJ:0.08
[Step] W-Obj: 18.2612 | k:1.1 th:0.053 xi:0.84 rho:-0.29 v0:0.02 | L:0.10 muJ:-0.01 sJ:0.08
[Step] W-Obj: 18.1951 | k:1.1 th:0.054 xi:0.84 rho:-0.29 v0:0.02 | L:0.10 muJ:-0.01 sJ:0.08
[Step] W-Obj: 18.0875 | k:1.1 th:0.054 xi:0.84 rho:-0.29 v0:0.02 | L:0.10 muJ:-0.01 sJ:0.08
[Step] W-Obj: 17.5151 | k:1.1 th:0.049 xi:0.86 rho:-0.30 v0:0.02 | L:0.08 muJ:-0.01 sJ:0.07
[Step] W-Obj: 17.5047 | k:1.1 th:0.049 xi:0.86 rho:-0.30 v0:0.02 | L:0.08 muJ:-0.01 sJ:0.07
[Step] W-Obj: 17.5046 | k:1.1 th:0.049 xi:0.86 rho:-0.30 v0:0.02 | L:0.08 muJ:-0.01 sJ:0.07
[Step] W-Obj: 17.5029 | k:1.1 th:0.049 xi:0.86 rho:-0.30 v0:0.02 | L:0.08 muJ:-0.01 sJ:0.07
[Step] W-Obj: 17.4081 | k:1.1 th:0.050 xi:0.85 rho:-0.30 v0:0.02 | L:0.08 muJ:-0.01 sJ:0.07
[Step] W-Obj: 17.3569 | k:1.1 th:0.052 xi:0.85 rho:-0.30 v0:0.02 | L:0.08 muJ:-0


Done in 88.54s | RMSE: 18.4587
  kappa   : 1.3040
  theta   : 0.0499
  xi      : 0.8423
  rho     : -0.4661
  v0      : 0.0232
  lamb    : 0.0609
  mu_j    : -0.0614
  sigma_j : 0.0755

MC_accel | S0: 6864.99 | RMSE: 36.3131
--------------------------------------------------------------------------------
     T      K Type  Market   Model  Spread  IV_Mkt
0.0414 5400.0  PUT   1.475  4.3739    0.35  0.4869
0.0414 6175.0  PUT   7.150  1.8967    0.70  0.3067
0.0414 6550.0  PUT  26.600 11.9326    0.80  0.2356
0.0414 6690.0  PUT  45.850 28.6054    0.90  0.2087
0.0441 5800.0  PUT   3.450  2.3859    0.70  0.3875


36.31313946241176

## Results comparison

In [17]:
results = {
    'Ana (naive)': res_naive,
    'Ana (accel)': res_fast,
    'MC  (naive)': res_mc_params,
    'MC  (accel)': res_mc_fast,
}
rows = []
for name, r in results.items():
    row = {'Method': name}
    for k in PARAM_KEYS:
        row[k] = f"{r[k]:.4f}"
    row['RMSE'] = f"{r.get('rmse', 0):.4f}"
    rows.append(row)

print(pd.DataFrame(rows).set_index('Method').to_string())

              kappa   theta      xi      rho      v0    lamb     mu_j sigma_j     RMSE
Method                                                                                
Ana (naive)  1.2848  0.0678  1.0694  -0.7938  0.0247  0.5334  -0.1222  0.0489   6.9639
Ana (accel)  1.4689  0.0490  0.7146  -0.7361  0.0267  0.0525  -0.4997  0.2263   7.2940
MC  (naive)  1.0147  0.0790  0.8247  -0.7283  0.0222  0.1364  -0.2503  0.1139  10.4419
MC  (accel)  1.3040  0.0499  0.8423  -0.4661  0.0232  0.0609  -0.0614  0.0755  18.4587
